# Milestone 2 — Unified Music Similarity Evaluation

Compare **original 518-dimensional FMA features**, **16-dimensional PCA**, and **16-dimensional feature-based autoencoder embeddings** under the **same retrieval protocol**: test queries against training candidates, cosine nearest neighbors, `k=5` and `k=10`. Evaluate genre consistency, cross-genre rate, tag overlap and tag coverage.

**Prerequisites:** commit `scripts/pca_baseline.py`, `scripts/retrieval.py`, and `scripts/evaluate.py` to your repository. The processed `training.npz`, `validation.npz`, `test.npz`, and `metadata.csv` must exist. The autoencoder embeddings must be exported separately from the **same trained autoencoder** using the instructions below. Do not interpret missing tags as zero similarity.

**Autoencoder:** trains automatically if `data/processed/ae_embeddings.npz` is missing. Uses the shared standalone autoencoder architecture but the existing repository splits.

## 1. Set up repository and imports

In [ ]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_NAME = "music-similarity-representation-learning"
REPO_URL = "https://github.com/samiyaalizaidi/music-similarity-representation-learning.git"

if "google.colab" in sys.modules:
    PROJECT_ROOT = Path("/content") / REPO_NAME
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "data" / "processed").is_dir()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Run this notebook from inside your repository.")

PROJECT_ROOT = PROJECT_ROOT.resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.pca_baseline import fit_pca, transform_pca
from scripts.retrieval import retrieve_neighbors
from scripts.evaluate import evaluate_retrieval

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "milestone2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
K_VALUES = [5, 10]
np.random.seed(SEED)
print("Project:", PROJECT_ROOT)

## 2. Load aligned processed features and metadata

In [ ]:
def load_split(name):
    with np.load(DATA_DIR / f"{name}.npz", allow_pickle=False) as data:
        X = data["X"].astype(np.float32)
        ids = data["track_ids"].astype(np.int64)
    assert X.ndim == 2 and len(X) == len(ids)
    return X, ids

X_train, train_ids = load_split("training")
X_val, val_ids = load_split("validation")
X_test, test_ids = load_split("test")
metadata = pd.read_csv(DATA_DIR / "metadata.csv", index_col=0)
metadata.index = pd.to_numeric(metadata.index, errors="raise").astype(np.int64)
assert metadata.index.is_unique
assert set(train_ids).issubset(metadata.index)
assert set(test_ids).issubset(metadata.index)
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Metadata columns:", list(metadata.columns))

GENRE_COLUMN = "genre" if "genre" in metadata.columns else "genre_top"
if GENRE_COLUMN not in metadata.columns:
    raise KeyError("Expected genre or genre_top in metadata.csv")
TAGS_COLUMN = "tags" if "tags" in metadata.columns else ("track_tags" if "track_tags" in metadata.columns else "tags")
if TAGS_COLUMN not in metadata.columns:
    print("WARNING: No tags column. Tag overlap metrics will be missing (NaN).")

## 3. Original-feature and PCA representations

In [ ]:
embeddings = {
    "original": {"train": X_train, "test": X_test}
}
pca = fit_pca(X_train, n_components=16, seed=SEED)
embeddings["pca"] = {
    "train": transform_pca(pca, X_train),
    "test": transform_pca(pca, X_test),
}
print("PCA retained variance:", round(float(pca.explained_variance_ratio_.sum()), 4))

## 4. Train and export autoencoder embeddings

This section uses the autoencoder architecture from the shared standalone script, but **does not use its separate data splitting or preprocessing**. It trains on the repository's `training.npz`, validates on `validation.npz`, and produces track-ID-aligned embeddings for the unified retrieval comparison. If the embeddings file already exists, training is skipped. A newly trained model is not the same fitted model as the previous Milestone 1 run; compare results as a reproducible rerun rather than claiming identical weights.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class AutoencoderConfig:
    input_dim: int
    latent_dim: int = 16
    hidden_dims: list = field(default_factory=lambda: [256, 128, 64])
    dropout: float = 0.1
    epochs: int = 50
    batch_size: int = 256
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5          # AdamW decoupled weight decay
    patience: int = 8                   # early-stopping patience (epochs)
    lr_patience: int = 5                # ReduceLROnPlateau patience (epochs)
    lr_factor: float = 0.5              # LR multiplier on plateau
    device: str = "cpu"
    seed: int = 42


class AutoencoderRepresentation:
    """Symmetric fully-connected autoencoder; the bottleneck is the embedding."""

    name = "autoencoder"

    def __init__(self, cfg: AutoencoderConfig):
        import torch
        import torch.nn as nn

        self.cfg = cfg
        torch.manual_seed(cfg.seed)
        self.device = torch.device(cfg.device)
        self.history = {"train_loss": [], "val_loss": []}

        def mlp(dims, final_activation):
            layers = []
            for i in range(len(dims) - 1):
                is_last = i == len(dims) - 2
                layers.append(nn.Linear(dims[i], dims[i + 1]))
                if not is_last or final_activation:
                    layers.append(nn.BatchNorm1d(dims[i + 1]))
                    layers.append(nn.ReLU(inplace=True))
                    if cfg.dropout > 0:
                        layers.append(nn.Dropout(cfg.dropout))
            return nn.Sequential(*layers)

        enc_dims = [cfg.input_dim, *cfg.hidden_dims, cfg.latent_dim]
        dec_dims = [cfg.latent_dim, *reversed(cfg.hidden_dims), cfg.input_dim]

        class _AE(nn.Module):
            def __init__(self):
                super().__init__()
                self.encoder = mlp(enc_dims, final_activation=False)
                self.decoder = mlp(dec_dims, final_activation=False)

            def encode(self, x):
                return self.encoder(x)

            def forward(self, x):
                return self.decoder(self.encoder(x))

        self.model = _AE().to(self.device)

    def fit(self, X_train, X_val=None):
        import torch
        import torch.nn as nn
        from torch.utils.data import DataLoader, TensorDataset

        cfg = self.cfg
        loader = DataLoader(
            TensorDataset(torch.as_tensor(X_train, dtype=torch.float32)),
            batch_size=cfg.batch_size, shuffle=True,
            drop_last=X_train.shape[0] > cfg.batch_size)  # BatchNorm needs >1 sample
        opt = torch.optim.AdamW(self.model.parameters(), lr=cfg.learning_rate,
                                weight_decay=cfg.weight_decay)
        # Decay the LR when validation loss plateaus. Kept separate from (and
        # complementary to) early stopping: the scheduler reshapes the descent,
        # early stopping decides when to quit and which weights to keep.
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=cfg.lr_factor, patience=cfg.lr_patience)
        criterion = nn.MSELoss()

        best_val = float("inf")
        best_state = None
        stale = 0
        for _ in range(cfg.epochs):
            self.model.train()
            running, seen = 0.0, 0
            for (batch,) in loader:
                batch = batch.to(self.device)
                opt.zero_grad()
                loss = criterion(self.model(batch), batch)
                loss.backward()
                opt.step()
                running += loss.item() * batch.size(0)
                seen += batch.size(0)
            self.history["train_loss"].append(running / max(seen, 1))

            if X_val is not None and len(X_val) > 0:
                val = self._eval_loss(X_val)
                self.history["val_loss"].append(val)
                scheduler.step(val)
                if val < best_val - 1e-6:
                    best_val = val
                    best_state = {k: v.detach().cpu().clone()
                                  for k, v in self.model.state_dict().items()}
                    stale = 0
                else:
                    stale += 1
                    if cfg.patience > 0 and stale >= cfg.patience:
                        break
        if best_state is not None:
            self.model.load_state_dict(best_state)
        return self

    def _eval_loss(self, X):
        import torch
        import torch.nn as nn
        self.model.eval()
        with torch.no_grad():
            t = torch.as_tensor(X, dtype=torch.float32, device=self.device)
            return nn.MSELoss()(self.model(t), t).item()

    def transform(self, X):
        import torch
        self.model.eval()
        with torch.no_grad():
            t = torch.as_tensor(X, dtype=torch.float32, device=self.device)
            return self.model.encode(t).cpu().numpy()

    def reconstruction_error(self, X):
        return self._eval_loss(X)



ae_path = DATA_DIR / "ae_embeddings.npz"
model_path = RESULTS_DIR / "autoencoder_state.pt"

if not ae_path.exists():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    cfg = AutoencoderConfig(
        input_dim=X_train.shape[1],
        latent_dim=16,
        hidden_dims=[256, 128, 64],
        epochs=50,
        device=device,
        seed=SEED,
    )
    ae = AutoencoderRepresentation(cfg)
    print(f"Training autoencoder on {device}...")
    ae.fit(X_train, X_val)
    print(f"Test reconstruction MSE: {ae.reconstruction_error(X_test):.4f}")
    np.savez_compressed(
        ae_path,
        train=ae.transform(X_train).astype(np.float32),
        test=ae.transform(X_test).astype(np.float32),
        train_ids=train_ids,
        test_ids=test_ids,
    )
    torch.save({
        "state_dict": ae.model.state_dict(),
        "config": vars(cfg),
        "history": ae.history,
    }, model_path)
    print("Saved:", ae_path)
    print("Saved model:", model_path)
else:
    print("Using existing embeddings:", ae_path)

with np.load(ae_path, allow_pickle=False) as saved:
    ae_train = saved["train"].astype(np.float32)
    ae_test = saved["test"].astype(np.float32)
    ae_train_ids = saved["train_ids"].astype(np.int64)
    ae_test_ids = saved["test_ids"].astype(np.int64)

if not np.array_equal(ae_train_ids, train_ids):
    raise ValueError("Autoencoder training IDs do not match processed training split")
if not np.array_equal(ae_test_ids, test_ids):
    raise ValueError("Autoencoder test IDs do not match processed test split")
if ae_train.shape[0] != len(X_train) or ae_test.shape[0] != len(X_test):
    raise ValueError("Autoencoder embedding row counts do not match data splits")

embeddings["autoencoder"] = {"train": ae_train, "test": ae_test}
print("Autoencoder embedding shapes:", ae_train.shape, ae_test.shape)


## 5. Same retrieval and metrics for every representation

In [ ]:
all_metrics, all_retrievals, all_detailed = [], [], []
for name, data in embeddings.items():
    neighbors = retrieve_neighbors(
        query_embeddings=data["test"],
        candidate_embeddings=data["train"],
        query_ids=test_ids,
        candidate_ids=train_ids,
        k=max(K_VALUES),
    )
    neighbors["representation"] = name
    all_retrievals.append(neighbors)
    for k in K_VALUES:
        metrics, detailed = evaluate_retrieval(
            neighbors, metadata, k=k,
            genre_column=GENRE_COLUMN, tags_column=TAGS_COLUMN,
        )
        metrics["representation"] = name
        all_metrics.append(metrics)
        detailed["representation"] = name
        detailed["k"] = k
        all_detailed.append(detailed)

comparison = pd.DataFrame(all_metrics)
retrieval_results = pd.concat(all_retrievals, ignore_index=True)
detailed_results = pd.concat(all_detailed, ignore_index=True)
display(comparison.round(4))
comparison.to_csv(RESULTS_DIR / "comparison.csv", index=False)
retrieval_results.to_csv(RESULTS_DIR / "retrieval_results.csv", index=False)
detailed_results.to_csv(RESULTS_DIR / "detailed_results.csv", index=False)
print("Saved tables to", RESULTS_DIR)

## 6. Plots

In [ ]:
results_k10 = comparison.query("k == 10").set_index("representation")
ax = results_k10[["genre_consistency", "cross_genre_rate"]].plot.bar(figsize=(9, 5), rot=0)
ax.set(title="Genre consistency vs cross-genre retrieval (k=10)", ylabel="Fraction", ylim=(0, 1))
plt.tight_layout()
plt.savefig(RESULTS_DIR / "genre_comparison.png", dpi=200)
plt.show()

if results_k10["tag_coverage"].gt(0).any():
    ax = results_k10[["tag_overlap", "cross_genre_tag_overlap"]].plot.bar(figsize=(9, 5), rot=0)
    ax.set(title="Tag overlap on evaluable pairs (k=10)", ylabel="Jaccard similarity")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "tag_comparison.png", dpi=200)
    plt.show()
else:
    print("No evaluable tag pairs; inspect metadata and tag coverage.")

## 7. Cross-genre examples and metadata coverage

In [ ]:
examples = detailed_results.query("representation == 'autoencoder' and k == 10").copy()
examples = examples[examples["cross_genre"] == 1.0].copy()
for column in ["title", "artist"]:
    if column in metadata.columns:
        examples[f"query_{column}"] = examples["query_id"].map(metadata[column])
        examples[f"neighbor_{column}"] = examples["neighbor_id"].map(metadata[column])
cols = [c for c in ["query_id", "query_title", "query_artist", "query_genre", "neighbor_id", "neighbor_title", "neighbor_artist", "neighbor_genre", "cosine_similarity", "tag_overlap"] if c in examples.columns]
display(examples[cols].head(20))
examples.to_csv(RESULTS_DIR / "cross_genre_examples.csv", index=False)
display(comparison[["representation", "k", "tag_coverage", "cross_genre_tag_coverage"]].round(4))

## Interpretation reminders

- A higher **cross-genre rate** is not automatically better musical similarity.
- Compare **tag overlap** only where metadata is available; include tag coverage.
- All three methods now use **the same test queries and training candidate tracks**.
- The precomputed-feature models do **not** directly learn from MP3 waveforms; this milestone follows the originally proposed feature-based comparison.
- For qualitative evaluation, listen to selected cross-genre neighbor pairs and record shared rhythm, timbre, instrumentation, or energy.